# [FitNets: Hints for Deep Nets](https://arxiv.org/pdf/1412.6550)

## Introduction

Depth is desired because it enables hierarchical feature reuse and often improves generalization, while thinness is desirable for computational efficiency.

**Gap:** Standard distillation focuses on matching final output distributions, which provides limited guidance for learning intermediate representations, making it difficult to train deep or narrow student networks effectively.

**Improvement:** FitNets extend knowledge distillation (KD) by introducing **intermediate-level supervision (“hints”)** from the teacher to guide student representations at hidden layers, rather than relying only on matching final outputs. This enables better training of deeper and thinner student models.

## Knowledge Distillation
$$
P_T^\tau = \operatorname{softmax}\left(\frac{a_T}{\tau}\right),
\qquad
P_S^\tau = \operatorname{softmax}\left(\frac{a_S}{\tau}\right)
$$

$$
\mathcal{L}_{KD}(W_S)
=
\mathcal{H}(y_{\text{true}}, P_S)
+
\lambda \, \mathcal{H}(P_T^\tau, P_S^\tau)
$$

## Hint-based Training
$$
\mathcal{L}_{HT}(W_{\text{Guided}}, W_r)
=
\frac{1}{2}
\left\|
u_h(x; W_{\text{Hint}})
-
r(v_g(x; W_{\text{Guided}}); W_r)
\right\|_2^2
$$

## Approach

1. **Hint-based training:** (Pre-)Train a chosen intermediate student layer to match a corresponding teacher hidden layer.
2. **Standard Distillation:** Train the full student model using soft targets from the teacher’s output distribution.

## Result

Shows that **including hint-based training** with distillation (0.51% error rate on MNIST) empirically performs better than distillation by itself (0.65% error rate on MNIST).

## Application

In [9]:
import torch
from torch import nn
import torch.nn.functional as F

# Large (complex) model with regularization (dropout)
class LargeNet(nn.Module):
    def __init__(self, input_size=(28,28), output_size=10, num_neurons=800, p=0.20):
        super(LargeNet, self).__init__()
        input_size = np.prod(input_size)
        self.fc1 = nn.Linear(input_size, num_neurons)
        self.fc2 = nn.Linear(num_neurons, num_neurons)
        self.out = nn.Linear(num_neurons, output_size)
        self.dropout = nn.Dropout(p)

    def forward(self, x, intermediate_layer=None):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        if intermediate_layer == 1:
            return x
        
        x = self.fc2(x)
        x = F.relu(x)
        x = self.dropout(x)
        if intermediate_layer == 2:
            return x
        
        logits = self.out(x)
        return logits

# Small (distilled) model without regularization
class FitNet(nn.Module):
    def __init__(self, input_size=(28,28), output_size=10, num_neurons=50):
        super(FitNet, self).__init__()
        input_size = np.prod(input_size)
        self.fc1 = nn.Linear(input_size, num_neurons)
        self.fc2 = nn.Linear(num_neurons, num_neurons)
        self.fc3 = nn.Linear(num_neurons, num_neurons)
        self.fc4 = nn.Linear(num_neurons, num_neurons)
        self.out = nn.Linear(num_neurons, output_size)

    def forward(self, x, intermediate_layer=None):
        x = self.fc1(x)
        x = F.relu(x)
        if intermediate_layer == 1:
            return x
        
        x = self.fc2(x)
        x = F.relu(x)
        if intermediate_layer == 2:
            return x

        x = self.fc3(x)
        x = F.relu(x)
        if intermediate_layer == 3:
            return x

        x = self.fc4(x)
        x = F.relu(x)
        if intermediate_layer == 4:
            return x
        
        logits = self.out(x)
        return logits

In [10]:
# Data
from torchvision import datasets, transforms

# Jitter 2 pixels
jitter = 2 / 28
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomAffine(degrees=0, translate=(jitter, jitter)),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Loading MNIST data
train_data = datasets.MNIST(root='data', train=True, download=True, transform=train_transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=test_transform)

# Create data loaders
BATCH_SIZE = 128
train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               RandomAffine(degrees=[0.0, 0.0], translate=(0.07142857142857142, 0.07142857142857142))
           )

In [11]:
from tqdm.notebook import tqdm
import copy

def train_model(model, optimizer, num_epochs=20, gamma=0.95):
    model = model.to(device)
    
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

    best_model_state = None
    best_model_errors = float("inf")
    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        running_loss = 0.0
    
        progress_bar = tqdm(total=len(train_loader))
    
        for inputs, labels in train_loader:
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)
            optimizer.zero_grad()  # Zero the gradients
            
            outputs = model(inputs)  # Forward pass
            loss = criterion(outputs, labels)  # Compute the loss
            loss.backward()  # Backward pass
            optimizer.step()  # Update the weights
            
            running_loss += loss.item() * inputs.size(0)
            
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
        scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        all_preds, all_labels = test_model(model)
        num_instances = len(all_preds)
        correct = (all_preds == all_labels).sum()
        accuracy =  correct / num_instances
        errors = num_instances - correct
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy}, Errors: {errors}, next LR: {next_lr}")

        # Save best model
        if errors < best_model_errors:
            best_model_errors = errors
            best_model_state = copy.deepcopy(model.state_dict())
            print(f"Saved new best model with {best_model_errors} test errors")

    # Load best model back
    model.load_state_dict(best_model_state)
    print(f"Loaded best model with {best_model_errors} test errors")
    return model

def test_model(model):
    model.eval()
    
    all_preds = []
    all_labels = []

    progress_bar = tqdm(total=len(test_loader))

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.view(inputs.shape[0], -1).to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            preds = outputs.argmax(dim=-1)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

            progress_bar.update(1)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return all_preds, all_labels

def load_model(model, path, device):
    state_dict = torch.load(path)
    model.load_state_dict(state_dict)
    return model.to(device)

In [12]:
import numpy as np 

LR = 1e-1
MOMENTUM = 0.9
large_model = LargeNet()
optimizer = torch.optim.SGD(large_model.parameters(), lr=LR, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [13]:
# large_model = train_model(large_model, optimizer, num_epochs=50)

large_model = load_model(large_model, path='ckpts/large_model.pt', device=device)
all_preds, all_labels = test_model(large_model)

accuracy = (all_preds == all_labels).float().mean().item()
errors = (all_preds != all_labels).sum().item()
print(f"Accuracy: {accuracy:.4f}, Errors: {errors}")

C:\Users\angel\AppData\Local\Temp\ipykernel_20172\2572169861.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path)


  0%|          | 0/79 [00:00<?, ?it/s]

Accuracy: 0.9935, Errors: 65


In [14]:
def hint_model(small_model, large_model, small_layer, large_layer, regressor, optimizer, num_epochs=15, gamma=0.95):
    small_model = small_model.to(device)
    large_model = large_model.to(device)
    large_model.eval()

    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

    loss_fn = nn.MSELoss()
    
    for epoch in range(num_epochs):
        small_model.train()  # Set the small_model to training mode
        running_loss = 0.0
    
        progress_bar = tqdm(total=len(train_loader))
    
        for inputs, labels in train_loader:
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)
            optimizer.zero_grad()  # Zero the gradients

            # Forward passes
            outputs_small = small_model(inputs, small_layer)
            outputs_small = regressor(outputs_small)
            outputs_large = large_model(inputs, large_layer)

            # Compute the loss
            loss = loss_fn(outputs_small, outputs_large)
            
            loss.backward()  # Backward pass
            optimizer.step()  # Update the weights
            
            running_loss += loss.item() * inputs.size(0)
            
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
        scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, next LR: {next_lr}")

In [15]:
def distil_model(small_model, large_model, optimizer, num_epochs=15, alpha=0.8, T=20, gamma=0.95):
    small_model = small_model.to(device)
    large_model = large_model.to(device)
    large_model.eval()

    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    for epoch in range(num_epochs):
        small_model.train()  # Set the small_model to training mode
        running_loss = 0.0
    
        progress_bar = tqdm(total=len(train_loader))
    
        for inputs, labels in train_loader:
            inputs, labels = inputs.view(inputs.shape[0], -1).to(device), labels.to(device)
            optimizer.zero_grad()  # Zero the gradients

            # Forward passes
            outputs_small = small_model(inputs)
            outputs_large = large_model(inputs)
            
            with torch.no_grad():
                teacher_probs = F.softmax(outputs_large / T, dim=1)
            student_log_probs = F.log_softmax(outputs_small / T, dim=1)

            # Compute the loss
            distil_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (T * T)
            loss = alpha * criterion(outputs_small, labels) + (1 - alpha) * distil_loss
            
            loss.backward()  # Backward pass
            optimizer.step()  # Update the weights
            
            running_loss += loss.item() * inputs.size(0)
            
            progress_bar.update(1)
            progress_bar.set_description(f"Loss: {loss.item():.4f}")
        scheduler.step()

        next_lr = optimizer.param_groups[0]["lr"]
        
        all_preds, all_labels = test_model(small_model)
        num_instances = len(all_preds)
        correct = (all_preds == all_labels).sum()
        accuracy =  correct / num_instances
        errors = num_instances - correct
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy}, Errors: {errors}, next LR: {next_lr}")

In [16]:
LR = 1e-1
MOMENTUM = 0.9
small_model = FitNet()
optimizer = torch.optim.SGD(small_model.parameters(), lr=LR, momentum=MOMENTUM)
criterion = nn.CrossEntropyLoss()

In [23]:
distil_model(small_model, large_model, optimizer, num_epochs=30)

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/30, Loss: 3.5324, Accuracy: 0.9223999977111816, Errors: 776, next LR: 0.095


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

...

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 29/30, Loss: 0.4041, Accuracy: 0.980400025844574, Errors: 196, next LR: 0.022593554099256556


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 30/30, Loss: 0.3922, Accuracy: 0.9810000061988831, Errors: 190, next LR: 0.021463876394293726


In [17]:
small_layer, large_layer = 3, 2
small_dim = getattr(small_model, f"fc{small_layer}").out_features
large_dim = getattr(large_model, f"fc{large_layer}").out_features

small_model = FitNet()
regressor = nn.Linear(small_dim, large_dim).to(device)
optimizer = torch.optim.SGD(list(small_model.parameters()) + list(regressor.parameters()), lr=LR, momentum=MOMENTUM)
regressor

Linear(in_features=50, out_features=800, bias=True)

In [18]:
small_model = load_model(small_model, path='ckpts/hinted_small_model.pth', device=device)

# hint_model(small_model, large_model, small_layer=small_layer, large_layer=large_layer, regressor=regressor, optimizer=optimizer, num_epochs=30)

C:\Users\angel\AppData\Local\Temp\ipykernel_20172\2572169861.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path)


In [19]:
LR = 1e-2
optimizer = torch.optim.SGD(small_model.parameters(), lr=LR, momentum=MOMENTUM)

In [20]:
distil_model(small_model, large_model, optimizer, num_epochs=30)

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 1/30, Loss: 1.9098, Accuracy: 0.9603000283241272, Errors: 397, next LR: 0.0095


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

...

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 29/30, Loss: 0.3081, Accuracy: 0.9846000075340271, Errors: 154, next LR: 0.0022593554099256553


  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

Epoch 30/30, Loss: 0.3075, Accuracy: 0.9843999743461609, Errors: 156, next LR: 0.0021463876394293723
